# 04 — Training trajectories and DEER answer-fix runs

One notebook, three jobs. Each Colab session runs **one** job (set `JOB` in step 5):

| `JOB` | What it runs | Time on a T4 |
|---|---|---|
| `gsm8k_train` | never-exit trajectories on 1,000 GSM8K **train** problems (4 chunks of 250) | ~1.5 h |
| `math_train` | never-exit trajectories on 500 MATH **train** problems (2 chunks of 250) | ~2 h |
| `answer_fix` | live DEER τ = 0.95 on the first 100 GSM8K and MATH-500 test problems with `--final_answer trial` and `boxed`, plus full reasoning with `boxed` | ~1 h |

Never-exit runs (τ = 1.1) log a trial answer and its confidence at every "Wait" checkpoint: the training data for
the adaptive stopping methods. The answer-fix runs test whether keeping DEER's trial answer (or prompting the
final answer with `\boxed{`) recovers the accuracy that DEER loses when it rewrites its answer.

**How to run:** T4 runtime → set `JOB` in step 5 → *Runtime → Run all*. Keep the tab open.
Runs are split into chunks and each finished chunk is backed up to Drive. If Colab disconnects, reconnect and
*Run all* again: finished chunks are restored and skipped.

## 1. GPU check
*Runtime → Change runtime type → T4 GPU* first. This should print `Tesla T4`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Google Drive backup
Outputs are copied to Drive after every run, so a Colab disconnect loses nothing.
If you reconnect, just run all cells again: finished runs are restored from Drive and skipped.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Get the code

In [ ]:
REPO = "https://github.com/PeramVishnuSree/Adaptive-Early-Exit-for-Efficient-Reasoning-Reproducing-and-Extending-DEER.git"
!git clone -q $REPO /content/repo || (cd /content/repo && git pull -q)
%cd /content/repo/Vishnu-evaluation
!git log --oneline -1

## 4. Install (~3–5 min)
vLLM 0.26.0 matches Colab's Python 3.13 / torch 2.11. Exit logic and decoding settings are the official DEER ones.

In [ ]:
!pip install -q "vllm==0.26.0" nltk
!pip install -q "antlr4-python3-runtime==4.11.0" "latex2sympy2_extended[antlr4_11_0]" regex word2number

Check the install. Expect `vllm 0.26.0 | torch 2.11.0+... | cuda True`, then a few generated words.
The first model load takes a few minutes.

In [ ]:
!python -c "import vllm, torch, sympy; print('vllm', vllm.__version__, '| torch', torch.__version__, '| sympy', sympy.__version__, '| cuda', torch.cuda.is_available())"
!python -c "from vllm import LLM, SamplingParams; llm = LLM('deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', dtype='float16', max_model_len=4096, gpu_memory_utilization=0.8); print(repr(llm.generate(['What is 2+3?'], SamplingParams(max_tokens=20, temperature=0))[0].outputs[0].text))"

**Only if the check fails with a CUDA error** (`undefined symbol`, `libcudart.so.12`, ...): set `FIX_CUDA = True`,
run this cell, then *Runtime → Restart session*, and run the check cell again.

In [ ]:
FIX_CUDA = False
if FIX_CUDA:
    !pip install -q "torch==2.11.0" "torchvision==0.26.0" "torchaudio==2.11.0" --index-url https://download.pytorch.org/whl/cu129

## 5. Settings
Set `JOB` for this session. Do not change the other values (they must match the checkpoint 3 runs).

In [ ]:
JOB = "gsm8k_train"     # one of: "gsm8k_train", "math_train", "answer_fix"

import os, glob, json, shutil
%cd /content/repo/Vishnu-evaluation
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
MAX_TOKENS = 8000       # same budget as checkpoint 3
DTYPE = "float16"       # T4 has no bf16
N_TEST = 100            # first 100 test problems (same as checkpoint 3)
CHUNK = 250             # training problems per chunk
COMMON = (f"--model_name_or_path {MODEL} --dataset_dir data --max_generated_tokens {MAX_TOKENS} "
          f"--think_ratio 0.6 --policy avg1 --dtype {DTYPE} --gpu-memory-utilization 0.9")

OUT = f"outputs/final/{JOB}"
DRIVE_DIR = f"/content/drive/MyDrive/deer_final/{JOB}"
os.makedirs(OUT, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)
shutil.copytree(DRIVE_DIR, OUT, dirs_exist_ok=True)   # restore finished chunks after a disconnect

RUNS = []   # (output file, dataset, extra arguments)
if JOB in ("gsm8k_train", "math_train"):
    n_total = sum(1 for _ in open(f"data/{JOB}/test.jsonl"))
    for part, off in enumerate(range(0, n_total, CHUNK)):
        RUNS.append((f"{OUT}/never_{JOB}_part{part}.jsonl", JOB,
                     f"--threshold 1.1 --max_judge_steps 50 --offset {off} --limit {CHUNK}"))
elif JOB == "answer_fix":
    for mode in ["trial", "boxed"]:
        for ds in ["gsm8k", "math"]:
            RUNS.append((f"{OUT}/deer_t0.95_{mode}_{ds}.jsonl", ds,
                         f"--threshold 0.95 --limit {N_TEST} --final_answer {mode}"))
    for ds in ["gsm8k", "math"]:   # full reasoning with the same boxed answer step (reference)
        RUNS.append((f"{OUT}/never_boxed_{ds}.jsonl", ds,
                     f"--threshold 1.1 --limit {N_TEST} --final_answer boxed"))
else:
    raise ValueError(f"unknown JOB {JOB!r}")

def done(path):
    """A run is finished once its .meta.json exists (written last)."""
    return os.path.exists(path[: -len(".jsonl")] + ".meta.json")

def backup():
    shutil.copytree(OUT, DRIVE_DIR, dirs_exist_ok=True)
    print("backed up to", DRIVE_DIR)

print(f"JOB={JOB}: {len(RUNS)} runs, already finished: {sum(done(r[0]) for r in RUNS)}")
for r in RUNS:
    print("  ", "done " if done(r[0]) else "to do", r[0])

## 6. Run
Each run is backed up to Drive as soon as it finishes. Progress lines are filtered to keep the output short.

In [ ]:
for out, ds, extra in RUNS:
    if done(out):
        print("skip (done):", out); continue
    print("running:", out)
    !python third_party/deer/vllm_deer_logged.py {COMMON} --dataset {ds} {extra} --output_file {out} 2>&1 | grep -v "threshold not reached\|Reached early exit\|reached thinking limit" | tail -4
    backup()

## 7. Merge chunks and score
Training jobs: the chunks are merged into one file (`never_<JOB>.jsonl`) before scoring.

In [ ]:
SCORE = []   # (file, dataset)
if JOB in ("gsm8k_train", "math_train"):
    parts = [r[0] for r in RUNS]
    assert all(done(p) for p in parts), "some chunks are not finished yet: run step 6 again"
    merged = f"{OUT}/never_{JOB}.jsonl"
    recs, wall = [], 0.0
    for p in parts:
        recs += [json.loads(l) for l in open(p)]
        wall += json.load(open(p[:-6] + ".meta.json"))["wall_time_s"]
    recs.sort(key=lambda r: r["idx"])
    with open(merged, "w") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    meta0 = json.load(open(parts[0][:-6] + ".meta.json"))
    json.dump({"wall_time_s": wall, "n_questions": len(recs), "args": meta0["args"], "parts": len(parts)},
              open(merged[:-6] + ".meta.json", "w"), indent=2)
    print(f"merged {len(parts)} chunks -> {merged} ({len(recs)} problems, {wall / 60:.1f} min GPU)")
    SCORE = [(merged, JOB)]
else:
    SCORE = [(out, ds) for out, ds, _ in RUNS]

for f, ds in SCORE:
    !python scripts/score_run.py --input {f} --dataset {ds}
backup()

print(f"\n{'run':40s} {'n':>4s} {'acc%':>6s} {'out tok':>8s} {'exits':>6s} {'min':>6s}")
for f, _ in SCORE:
    s = json.load(open(f[:-6] + ".summary.json"))
    print(f"{s['input']:40s} {s['n']:4d} {100 * s['accuracy']:6.1f} {s['avg_output_tokens']:8.0f} "
          f"{s['exit_reasons'].get('confident', 0):6d} {(s.get('wall_time_s') or 0) / 60:6.1f}")

## 8. Save and send
Downloads `final_<JOB>.zip`. Send it to Vishnu (or share the Drive folder `deer_final/<JOB>`).

In [ ]:
zip_name = f"final_{JOB}.zip"
!zip -qr {zip_name} {OUT}
from google.colab import files
files.download(zip_name)